In [1]:
from sympy.ntheory import factorint
from tools import *

import matplotlib.pyplot as plt
import netCDF4 as nc
import numpy as np
import torchtt as tt

In [2]:
def bfs(meshname, ind_type):
    ds = nc.Dataset(meshname)
    if ind_type == 'cells':
        nodes_on_node = ds.variables['cellsOnCell'][:]
        visited = np.zeros(ds.dimensions['nCells'].size, dtype=int)
    elif ind_type == 'edges':
        nodes_on_node = ds.variables['edgesOnEdge'][:]
        visited = np.zeros(ds.dimensions['nEdges'].size, dtype=int)
    else:
        print(f'ind_type of {ind_type} not valid.')
        return
    # END if
    ds.close()

    root_ind = 0
        
    sorted_data = []
    queue = [(root_ind, 0)]
    visited[root_ind] = 1
    while queue:
        cur_ind, cur_depth = queue.pop(0)
        sorted_data.append((cur_ind, cur_depth))
    
        adjacent_inds = np.array(nodes_on_node[cur_ind][nodes_on_node[cur_ind] != 0]) - 1
        for ind in adjacent_inds:
            if not visited[ind]:
                queue.append((ind, cur_depth + 1))
                visited[ind] = 1
            # END if
        # END for
    # END while   
    ordered_inds = np.array([data[0] for data in sorted_data])
    ordered_levels = np.array([data[1] for data in sorted_data])

    return ordered_inds, ordered_levels
# END bfs()

In [3]:
mesh_num = 5
meshname = f'io/mesh_cvt_{mesh_num}.nc'


ordered_inds, ordered_levels = bfs(meshname, 'cells')
print(ordered_inds.size, ordered_levels.size)

if False:
    ds = nc.Dataset(meshname, 'a', format='NETCDF4')
    bfs_depth_name = 'bfs_depth_cell'
    try:
        ds.createVariable(bfs_depth_name, 'i', ('nCells'))
        print(f'created variable {bfs_depth_name}')
    except:
        print(f'variable {bfs_depth_name} already exists')
    # END try
    ds.variables[bfs_depth_name][ordered_inds] = ordered_levels
    ds.close()
# END if


ordered_inds, ordered_levels = bfs(meshname, 'edges')
print(ordered_inds.size, ordered_levels.size)

if False:
    ds = nc.Dataset(meshname, 'a', format='NETCDF4')
    bfs_depth_name = 'bfs_depth_edge'
    try:
        ds.createVariable(bfs_depth_name, 'i', ('nEdges'))
        print(f'created variable {bfs_depth_name}')
    except:
        print(f'variable {bfs_depth_name} already exists')
    # END try
    ds.variables[bfs_depth_name][ordered_inds] = ordered_levels
    ds.close()
# END if

10242 10242
30720 30720


In [14]:
def get_node_loc(ind, mesh_data):
    return np.array([mesh_data['xnode'][ind],
                     mesh_data['ynode'][ind],
                     mesh_data['znode'][ind]])
# END get_node_loc()


def order_by_angle(root_ind, root_vec, adjacent, mesh_data):
    root_pt = get_node_loc(root_ind, mesh_data)
    
    angles = np.zeros(adjacent.size)
    for i, ind in enumerate(adjacent):
        pt = get_node_loc(ind, mesh_data)
        vec = pt - root_pt
        
        angle = np.arccos(np.dot(root_vec, vec) / np.linalg.norm(vec))
        if vec[1] < 0:
            angle = 2 * np.pi - angle
        # END if
        angles[i] = angle
    # END for
    
    sorted_inds = np.argsort(angles)
    return adjacent[sorted_inds], angles[sorted_inds]
# END order_by_angle


def dist_from_origin(vec, pt, mesh_data):
    origin = np.array([0, 0, 0])
    radius = mesh_data['radius']
    
    new_pt = pt + (radius / 10) * vec
    return np.linalg.norm(origin - new_pt)
# END dist_from_origin()


def get_next_in_spiral(prev_ind, cur_ind, candidate_inds, mesh_data):
    radius = mesh_data['radius']
    latnode = mesh_data['latnode']
    node_depth = mesh_data['node_depth']
    
    prev_pt = get_node_loc(prev_ind, mesh_data)
    cur_pt = get_node_loc(cur_ind, mesh_data)
    root_vec = cur_pt - prev_pt
    root_vec = root_vec / np.linalg.norm(root_vec)

    dists = np.zeros(len(candidate_inds))
    for i, ind in enumerate(candidate_inds):
        pt = get_node_loc(ind, mesh_data)
        vec = pt - prev_pt
        vec = vec / np.linalg.norm(vec)
        cross = np.cross(root_vec, vec)

        dists[i] = dist_from_origin(cross, pt, mesh_data)
    # END for
    inside_inds = candidate_inds[np.where(dists < radius)]
    inside_dists = dists[np.where(dists < radius)]
    outside_inds = candidate_inds[np.where(dists >= radius)]
    outside_dists = dists[np.where(dists >= radius)]

    # only take a cross pointing outside of the sphere if
    # there are none pointing inside available
    if inside_inds.size > 0:
        # get ind for cross closest to 1
        ind = inside_inds[np.argmin(np.abs(1 - inside_dists))]
    else:
        # get ind for cross closest to 1
        ind = outside_inds[np.argmin(np.abs(1 - outside_dists))]
    # END if

    return ind
# END get_next_in_spiral()


def spiral(meshname, ind_type='cells'):
    ### get mesh data
    ds = nc.Dataset(meshname)
    radius = ds.__dict__['sphere_radius']
    if ind_type == 'cells':
        xnode = ds.variables['xCell'][:]
        ynode = ds.variables['yCell'][:]
        znode = ds.variables['zCell'][:]
        latnode = ds.variables['latCell'][:]
        nodes_on_node = ds.variables['cellsOnCell'][:]
        node_depth = ds.variables['bfs_depth_cell'][:]
        nnodes = ds.dimensions['nCells'].size
    elif ind_type == 'edges':
        xnode = ds.variables['xEdge'][:]
        ynode = ds.variables['yEdge'][:]
        znode = ds.variables['zEdge'][:]
        latnode = ds.variables['latEdge'][:]
        nodes_on_node = ds.variables['edgesOnEdge'][:]
        node_depth = ds.variables['bfs_depth_edge'][:]
        nnodes = ds.dimensions['nEdges'].size
    else:
        print(f'ind_type of {ind_type} not valid.')
        return
    # END if
    ds.close()

    ### build meta dict for mesh data
    mesh_data  = {'xnode': xnode,
                  'ynode': ynode,
                  'znode': znode,
                  'latnode': latnode,
                  'nodes_on_node': nodes_on_node,
                  'node_depth': node_depth,
                  'nnodes': nnodes,
                  'radius': radius}

    ### step 0: init
    root_ind = 0
    sorted_inds = [root_ind]
    visited = np.zeros(nnodes, dtype=int)
    visited[root_ind] = 1

    ### step 1: manually get second node
    root_vec = np.array([1, 0, 0])
    adjacent_inds = np.array(nodes_on_node[root_ind][nodes_on_node[root_ind] != 0]) - 1
    adjacent_inds, angles = order_by_angle(root_ind, root_vec, adjacent_inds, mesh_data)

    sorted_inds.append(adjacent_inds[0])
    visited[adjacent_inds[0]] = 1

    while not np.all(visited):
        ### step 2: choose between two candidate cells within current depth
        cur_ind = sorted_inds[-1]
        cur_depth = node_depth[cur_ind]
        
        adjacent_inds = np.array(nodes_on_node[cur_ind][nodes_on_node[cur_ind] != 0]) - 1
        candidate_inds = adjacent_inds[np.where(node_depth[adjacent_inds] == cur_depth)]
        
        ind = get_next_in_spiral(sorted_inds[-2], cur_ind, candidate_inds, mesh_data)
        sorted_inds.append(ind)
        visited[ind] = 1
    
        ### step 3: add all remaining cells at depth 1
        cur_depth_inds = np.where(node_depth == cur_depth)[0]
        while not np.all(visited[cur_depth_inds]):
            cur_ind = sorted_inds[-1]
            # first get all adjacent inds
            adjacent_inds = np.array(nodes_on_node[cur_ind][nodes_on_node[cur_ind] != 0]) - 1
            # then, remove all nor from the current depth
            candidate_inds = adjacent_inds[np.where(node_depth[adjacent_inds] == cur_depth)]
            # then, only allow an ind that hasn't been visited (should only be 1 option)
            ind = candidate_inds[np.where(visited[candidate_inds] == 0)][0]
            sorted_inds.append(ind)
            visited[ind] = 1
        # END all
    
        ### step 4: move up to next depth level
        cur_ind = sorted_inds[-1]
        new_depth = node_depth[cur_ind] + 1
    
        adjacent_inds = np.array(nodes_on_node[cur_ind][nodes_on_node[cur_ind] != 0]) - 1
        candidate_inds = adjacent_inds[np.where(node_depth[adjacent_inds] == new_depth)]
        
        ind = get_next_in_spiral(sorted_inds[-2], cur_ind, candidate_inds, mesh_data)
        sorted_inds.append(ind)
        visited[ind] = 1
    # END while

    return np.array(sorted_inds)
# END spiral()

In [15]:
sorted_inds = spiral(meshname)

print(sorted_inds.size)
print(sorted_inds)

10242
[   0 2562 2563 ... 2569 2570    1]


In [6]:
if True:
    ds = nc.Dataset(meshname, 'a', format='NETCDF4')
    spiral_cell_name = 'test_spiral_order_cell'
    try:
        ds.createVariable(spiral_cell_name, 'i', ('nCells'))
        print(f'created variable {spiral_cell_name}')
    except:
        print(f'variable {spiral_cell_name} already exists')
    # END try
    ds.variables[spiral_cell_name][sorted_inds] = np.arange(ds.dimensions['nCells'].size)
    ds.close()
# END if

variable test_spiral_order_cell already exists


In [7]:
ds = nc.Dataset(meshname)
spiral_inds_cell = np.argsort(ds.variables['spiral_order_cell'])
ds.close()

print(np.where(sorted_inds == spiral_inds_cell))
print(np.all(sorted_inds == spiral_inds_cell))

(array([    0,     1,     2, ..., 10239, 10240, 10241], shape=(10242,)),)
True
